In [2]:
# UMLS Vaccine CUI Expansion (CUI → AUI → related AUI → CUI)

## Step 1: Imports and Setup
import requests
import time
from threading import Lock
from functools import wraps
from typing import List, Set, Dict
import concurrent.futures
from tqdm import tqdm

## Step 2: Rate Limiter
class RateLimiter:
    def __init__(self, max_calls_per_second=5):
        self.min_interval = 1.0 / max_calls_per_second
        self.last_called = 0
        self.lock = Lock()

    def wait(self):
        with self.lock:
            elapsed = time.time() - self.last_called
            wait_time = self.min_interval - elapsed
            if wait_time > 0:
                time.sleep(wait_time)
            self.last_called = time.time()

rate_limiter = RateLimiter(max_calls_per_second=10)

def rate_limited(rate_limiter: RateLimiter):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            rate_limiter.wait()
            return func(*args, **kwargs)
        return wrapper
    return decorator

## Step 3: Service Ticket Retrieval
def get_service_ticket(api_key: str) -> str:
    tgt_response = requests.post(
        "https://utslogin.nlm.nih.gov/cas/v1/api-key",
        data={"apikey": api_key}
    )
    tgt_url = tgt_response.text.split('action="')[1].split('"')[0]
    st_response = requests.post(tgt_url, data={"service": "http://umlsks.nlm.nih.gov"})
    return st_response.text.strip()

## Step 4: UMLS API Functions
@rate_limited(rate_limiter)
def get_auis_for_cui(cui: str, apikey: str) -> List[str]:
    url = f"https://uts-ws.nlm.nih.gov/rest/content/current/CUI/{cui}/atoms"
    ticket = get_service_ticket(apikey)
    r = requests.get(url, params={"ticket": ticket})
    r.raise_for_status()
    atoms = r.json().get("result", [])
    return [a["ui"] for a in atoms if "ui" in a]

@rate_limited(rate_limiter)
def get_related_auis(aui: str, apikey: str, relation_types: List[str]) -> List[str]:
    url = f"https://uts-ws.nlm.nih.gov/rest/content/current/Atom/{aui}/relations"
    ticket = get_service_ticket(apikey)
    r = requests.get(url, params={"ticket": ticket})
    r.raise_for_status()
    relations = r.json().get("result", [])
    return [
        rel["relatedId"]
        for rel in relations
        if rel.get("relationLabel") in relation_types and "relatedId" in rel
    ]

@rate_limited(rate_limiter)
def get_cui_from_aui(aui: str, apikey: str) -> str:
    url = f"https://uts-ws.nlm.nih.gov/rest/content/current/Atom/{aui}"
    ticket = get_service_ticket(apikey)
    r = requests.get(url, params={"ticket": ticket})
    r.raise_for_status()
    return r.json().get("result", {}).get("concept", {}).get("ui")

## Step 5: Expansion Pipeline
def expand_cuis_from_seed(seed_cuis: List[str], apikey: str) -> Set[str]:
    all_related_cuis = set()
    relation_types = ["RN", "CHD", "RO", "SY", "RQ"]

    for cui in tqdm(seed_cuis, desc="Seed CUIs"):
        try:
            auis = get_auis_for_cui(cui, apikey)
            for aui in tqdm(auis, leave=False, desc=f"AUI for {cui}"):
                related_auis = get_related_auis(aui, apikey, relation_types)
                for related_aui in related_auis:
                    related_cui = get_cui_from_aui(related_aui, apikey)
                    if related_cui and related_cui != cui:
                        all_related_cuis.add(related_cui)
        except Exception as e:
            print(f"Error processing CUI {cui}: {e}")

    return all_related_cuis

## Step 6: Run the Expansion
# Replace with your API key
apikey = "0c4d7175-6688-4c45-9a13-4c64d7354673"

# Start from these base vaccine CUIs
seed_cuis = ["C0042210", "C5399710"]

# Expand to related vaccine CUIs
expanded_cuis = expand_cuis_from_seed(seed_cuis, apikey)

print(f"Discovered {len(expanded_cuis)} related vaccine CUIs")
print(expanded_cuis)


Seed CUIs:   0%|          | 0/2 [00:00<?, ?it/s]

Seed CUIs:  50%|█████     | 1/2 [00:01<00:01,  1.14s/it]

Error processing CUI C0042210: 404 Client Error: Not Found for url: https://uts-ws.nlm.nih.gov/rest/content/current/Atom/A24665667/relations?ticket=ST-29335-41a25b4kqwme0dyzcv-cas


Seed CUIs: 100%|██████████| 2/2 [00:02<00:00,  1.06s/it]

Error processing CUI C5399710: 404 Client Error: Not Found for url: https://uts-ws.nlm.nih.gov/rest/content/current/Atom/A32340275/relations?ticket=ST-29339-41a25b4kqwme0dz046-cas
Discovered 0 related vaccine CUIs
set()
